In [7]:
# ── Cell 1: Install required packages ────────────────────────────────────────────
%pip install psitip anthropic pyomo gurobipy -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# ── Cell 2: Imports & global configuration ──────────────────────────────────
import os
import re
import json
import time
import glob
import difflib
import datetime
import pathlib
import importlib.util
from collections import Counter

import httpx
from psitip import *
from anthropic import Anthropic
import pandas as pd

PROJECT_DIR = r"C:\Users\User\OneDrive\ESCE520\Project"
RESULT_DIR = rf"{PROJECT_DIR}\Result"
os.environ["GRB_LICENSE_FILE"] = rf"{PROJECT_DIR}\Key\gurobi.lic"

# ── PSITIP setup ──────────────────────────────────────────────
PsiOpts.setting(solver="pyomo.gurobi", str_style="std")
X, Y, Z, W, U, V = rv("X, Y, Z, W, U, V")

# ── llm API setup ──────────────────────────────────────────
API_KEY_PATH = rf"{PROJECT_DIR}\Key\API_CLAUDE_KEY.txt"
with open(API_KEY_PATH) as f:
    api_key = f.read().strip()
# Default connect timeout is only 5 s; bump to 30 s to survive brief network blips.
client = Anthropic(api_key=api_key, timeout=httpx.Timeout(600.0, connect=30.0))
MODEL = "claude-haiku-4-5"

# ── API parameters ────────────────────────────────────────────
TEMPERATURE = 0.0
MAX_TOKENS  = 10000   # Non-Shannon proofs can be verbose; 

# ── Run mode ──────────────────────────────────────────────────
USE_BATCH   = True    # True → Batch API (~50% cheaper, async); False → Sequential
FORCE_RERUN = True    # True → ignore checkpoint and rerun ALL cases fresh

POLL_INTERVAL  = 60
BATCH_MAX_TOKS = MAX_TOKENS   # inherit from cell 2 so there's one place to change it

_client = client   # alias used by _run_pass batch path

_ts = datetime.datetime.now().strftime("%Y-%m-%d_%H%M%S") if FORCE_RERUN else datetime.date.today()


print("PSITIP loaded · llm client ready")
print(f"  Model : {MODEL}")
print(f"  MAX_TOKENS = {MAX_TOKENS}  TEMPERATURE = {TEMPERATURE}")
print(f"  Run mode    : {'BATCH' if USE_BATCH else 'SEQUENTIAL'}")
print(f"  Force rerun : {FORCE_RERUN}")
print(f"  Vars  : X, Y, Z, W, U, V")


PSITIP loaded · llm client ready
  Model : claude-haiku-4-5
  MAX_TOKENS = 10000  TEMPERATURE = 0.0
  Run mode    : BATCH
  Force rerun : True
  Vars  : X, Y, Z, W, U, V


In [9]:
# ── Cell 3: Scoring & proof-comparison functions ───────────────────────────────
# ══════════════════════════════════════════════════════════════
# Scoring weights & thresholds  (edit here to tune evaluation)
# ══════════════════════════════════════════════════════════════
_W_PSITIP_CONSEC = 0.50   # PSITIP LP consecutive-transition score
_W_AXIOM         = 0.50   # axiom recall (with compatibility map)


# ══════════════════════════════════════════════════════════════
# Response parsing
# ══════════════════════════════════════════════════════════════

_ABANDON_RE = re.compile(
    r'\b(?:let\s+me\s+(?:re-?derive|re-?do|re-?calculate|re-?think|try\s+again'
    r'|approach\s+this|reconsider|start\s+over|restart)'
    r'|I\s+need\s+to\s+(?:re-?start|re-?think|re-?derive|re-?calculate)'
    r'|this\s+approach\s+(?:isn\'t|is\s+not)\s+working'
    r'|actually,?\s+I\s+(?:made\s+an?\s+error|need\s+to\s+re)'
    r'|let\s+me\s+use\s+the\s+(?:correct|direct|clean|proper)\s+approach'
    r'|let\s+me\s+provide\s+the\s+clean\s+proof)',
    re.IGNORECASE,
)


def _build_user_prompt(label, assumption):
    if "  =>  " in label:
        ineq_part = label.split("  =>  ", 1)[1].strip()
    else:
        ineq_part = label
    assumption_line = assumption if assumption else "(none)"
    return f"INEQUALITY : {ineq_part}\nCONSTRAINTS: {assumption_line}"


def call_llm(label, assumption, system_prompt=None):
    """Send a single synchronous request to llm. Returns (text, usage)."""
    if system_prompt is None:
        system_prompt = SYSTEM_PROMPT_VERDICT
    msg = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=[{"type": "text", "text": system_prompt,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": _build_user_prompt(label, assumption)}],
    )
    text = "".join(b.text for b in msg.content if hasattr(b, "text")).strip()
    return text, msg.usage


def parse_llm_response(text, out_tok=None, max_tok=3000):
    """Extract VERDICT, TYPE, and Proof section from llm's response."""
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'```[\s\S]*?```', '\n', text)
    text = re.sub(r'`[^`]*`', '', text)

    verdict = "Unknown"
    verdict_labels = [r'VERDICT', r'VERDIC', r'RESULT', r'ANSWER', r'CONCLUSION']
    _labeled = []
    for lbl in verdict_labels:
        for m in re.finditer(rf'^{lbl}[:\-\s]*([^\n]+)', text, re.IGNORECASE | re.MULTILINE):
            cand = m.group(1).strip()
            v_m = re.search(r'\b(True|False|Yes|No)\b', cand, re.IGNORECASE)
            if v_m:
                v = v_m.group(1).lower()
                v_str = 'True' if v in ('true', 'yes') else 'False' if v in ('false', 'no') else None
                if v_str:
                    _labeled.append((m.start(), v_str))
    if _labeled:
        verdict = sorted(_labeled, key=lambda x: x[0])[-1][1]

    if verdict == 'Unknown':
        _concluding = []
        for m in re.finditer(r'(?:therefore|thus|conclusion|hence)[:,\s]*([^\n]+)', text, re.IGNORECASE):
            v_m = re.search(r'\b(True|False|Yes|No)\b', m.group(1), re.IGNORECASE)
            if v_m:
                v = v_m.group(1).lower()
                v_str = 'True' if v in ('true', 'yes') else 'False' if v in ('false', 'no') else None
                if v_str:
                    _concluding.append((m.start(), v_str))
        if _concluding:
            verdict = sorted(_concluding, key=lambda x: x[0])[-1][1]

    if verdict == 'Unknown':
        all_tf = re.findall(r'\b(True|False)\b', text[:300], re.IGNORECASE)
        if all_tf:
            verdict = all_tf[-1].capitalize()

    ptype = 'Unknown'
    for m in re.finditer(r'^TYPE[:\-\s]*(.+)$', text, re.IGNORECASE | re.MULTILINE):
        t = re.sub(r'^\*+\s*', '', m.group(1).strip())
        if t:
            ptype = t

    proof = None
    proof_starts = list(re.finditer(r'\bProof[:\s]*\n', text, re.IGNORECASE))
    if proof_starts:
        body = text[proof_starts[-1].end():]
        stop = re.search(r'\n(?:VERDICT|RESULT|ANSWER|CONCLUSION|END)[:\-\s]', body, re.IGNORECASE)
        proof = body[:stop.start()].strip() if stop else body.strip()
        proof = re.sub(
            r'^\s*VERDICT[:\-\s]*\S.*\n(?:TYPE[:\-\s]*.*\n)?(?:Proof[:\s]*\n)?',
            '', proof, flags=re.IGNORECASE
        ).strip()

    if not proof:
        proof = text.strip() if len(text.strip()) < 200 else None

    if out_tok is not None and out_tok >= max_tok:
        verdict = 'Unknown'
        return verdict, ptype, proof

    if proof and _ABANDON_RE.search(proof):
        _last_ab = list(_ABANDON_RE.finditer(proof))[-1]
        _line_end = proof.find('\n', _last_ab.end())
        if _line_end != -1:
            _remainder = proof[_line_end:].strip()
            if len(_remainder) >= 30:
                proof = _remainder
            elif verdict == 'True':
                verdict = 'Unknown'
        elif verdict == 'True':
            verdict = 'Unknown'

    return verdict, ptype, proof


def run_psitip(region, ctx_fn=None):
    """Run PSITIP on a region. Returns (bool_result, proof_str)."""
    if ctx_fn is not None:
        with ctx_fn():
            result = bool(region)
            proof  = str(region.proof(shorten=False)).strip() if result else ""
    else:
        result = bool(region)
        proof  = str(region.proof(shorten=False)).strip() if result else ""
    return result, proof


# ══════════════════════════════════════════════════════════════
# Proof comparison -- constants
# ══════════════════════════════════════════════════════════════

_COPY_VAR_RE = re.compile(r"\b(?:[A-Z]_\d+|D[A-Z](?:_\d+)?)\b|[A-Za-z]'")

_AXIOM_COMPAT = {
    'markov':       {'markov', 'dpi', 'H=0', 'I>=0'},
    'dpi':          {'dpi', 'markov'},
    'H=0':          {'H=0', 'markov', 'I>=0'},
    'chain_rule':   {'chain_rule'},
    'cond_reduces': {'cond_reduces', 'I>=0', 'I|>=0'},
    'I>=0':         {'I>=0', 'I|>=0', 'H>=0', 'H|>=0', 'cond_reduces'},
    'I|>=0':        {'I|>=0', 'I>=0', 'cond_reduces'},
    'H>=0':         {'H>=0', 'I>=0'},
    'H|>=0':        {'H|>=0', 'I>=0', 'I|>=0'},
    'copy_lemma':   {'copy_lemma', 'independence', 'H=0', 'I|>=0'},
    'independence': {'independence', 'copy_lemma'},
}

_SYM_CHECK = "\u2713"   # ✓
_SYM_CROSS = "\u2717"   # ✗
_SYM_DASH  = "\u2500"   # ─
_SYM_DBAR  = "\u2550"   # ═
_SYM_TL    = "\u250c"   # ┌
_SYM_PIPE  = "\u2502"   # │
_SYM_BL    = "\u2514"   # └

_SKIP_PREFIXES = (
    '{', '}', 'or ', 'substitute', 'introduce', 'identities',
    'under ', 'claim', 'steps:', '[introduce', 'disproof', 'counterexample',
    'lhs =', 'rhs =', 'violation',
)


# ══════════════════════════════════════════════════════════════
# Proof comparison -- helper functions
# ══════════════════════════════════════════════════════════════

def _sim(a: str, b: str) -> float:
    return difflib.SequenceMatcher(None, a, b).ratio()


def _norm_expr(expr: str) -> str:
    expr = re.sub(r'\s+', '', expr.strip().lower())
    expr = expr.replace('\u00b7', '*').replace('==', '=')

    def sort_h(m):
        inner = m.group(1).replace('+', ',')
        if '|' in inner:
            main, cond = inner.split('|', 1)
            mp = ','.join(sorted(v.strip() for v in main.split(',')))
            cp = ','.join(sorted(v.strip() for v in cond.split(',')))
            return 'h(' + mp + '|' + cp + ')'
        return 'h(' + ','.join(sorted(v.strip() for v in inner.split(','))) + ')'
    expr = re.sub(r'h\(([^)]+)\)', sort_h, expr)

    def sort_i(m):
        inner = m.group(1).replace('+', ',')
        if '|' in inner:
            main, cond = inner.rsplit('|', 1)
            cond_str = '|' + ','.join(sorted(v.strip() for v in cond.split(',')))
        else:
            main, cond_str = inner, ''
        parts = main.split(';', 1)
        if len(parts) == 2:
            a = ','.join(sorted(v.strip() for v in parts[0].split(',')))
            b = ','.join(sorted(v.strip() for v in parts[1].split(',')))
            a, b = (a, b) if a <= b else (b, a)
            return f'i({a};{b}{cond_str})'
        return m.group(0)
    expr = re.sub(r'i\(([^)]+)\)', sort_i, expr)
    return expr


def _has_copy_vars(text: str) -> bool:
    return bool(_COPY_VAR_RE.search(text))


def _parse_chain(proof_text):
    """Extract proof chain: [(relation, norm_expr, reason)]."""
    chain = []
    proof_text = (proof_text
        .replace('\u2264', '<=').replace('\u2265', '>=')
        .replace('\u2a7d', '<=').replace('\u2a7e', '>='))
    proof_text = re.sub(r'\$([^$]+)\$', r'\1', proof_text)
    raw_lines = proof_text.split('\n')
    idx = 0
    while idx < len(raw_lines):
        line = raw_lines[idx].strip()
        idx += 1
        if not line:
            continue
        line = re.sub(r'^[-\u2022*]\s+', '', line)
        line = re.sub(r'^(?:step\s*\d+[:.\s]*)', '', line, flags=re.IGNORECASE)
        line = re.sub(r'^\d+\.\s*(?:steps?:?\s*)?', '', line, flags=re.IGNORECASE)
        ll = line.lower()
        if any(ll.startswith(p) for p in _SKIP_PREFIXES):
            continue
        if ll.startswith('lp_verify'):
            break
        m = re.match(r'^([<>=]+)\s+(.+)', line)
        if m:
            rel = m.group(1).strip()
            rest = m.group(2).strip()
            while idx < len(raw_lines):
                nxt = raw_lines[idx].strip()
                nxt = (nxt.replace('\u2264', '<=').replace('\u2265', '>=')
                          .replace('\u2a7d', '<=').replace('\u2a7e', '>='))
                nxt = re.sub(r'\$([^$]+)\$', r'\1', nxt)
                if not nxt or re.match(r'^[<>=]', nxt):
                    break
                if any(nxt.lower().startswith(p) for p in _SKIP_PREFIXES):
                    break
                rest += ' ' + nxt
                idx += 1
            since_m = re.search(r'\(since\s+(.+?)\)\s*$', rest, re.IGNORECASE)
            if since_m:
                reason = since_m.group(1).strip()
                expr = rest[:since_m.start()].strip()
            else:
                paren_m = re.search(r'\(([^()]+)\)\s*$', rest)
                if paren_m:
                    _before_paren = rest[:paren_m.start()].rstrip()
                    _is_func_arg = bool(_before_paren) and _before_paren[-1].isalpha()
                if paren_m and not _is_func_arg:
                    reason = paren_m.group(1).strip()
                    expr = rest[:paren_m.start()].strip()
                else:
                    bare_m = re.search(r'\bsince\s+(.+)$', rest, re.IGNORECASE)
                    if bare_m and not re.search(r'[IHih]\(', bare_m.group(1)[:25]):
                        reason = bare_m.group(1).strip().rstrip(')')
                        expr = rest[:bare_m.start()].strip()
                    else:
                        reason = ''
                        expr = rest
            chain.append((rel, _norm_expr(expr), reason.lower()))
        elif re.search(r'[HIhi]\(', line) and not chain:
            candidate = re.sub(r'\(since\b.*$', '', line, flags=re.IGNORECASE).strip()
            standalone_m = re.search(r'(?<![A-Za-z])\s+\((?!.*[IHih]\()[^)]*\)\s*$', candidate)
            if standalone_m:
                candidate = candidate[:standalone_m.start()].strip()
            for _irel in ('<=', '>='):
                if _irel in candidate:
                    _lhs = candidate.split(_irel, 1)[0].strip()
                    if re.search(r'[HIhi]\(', _lhs):
                        candidate = _lhs
                    break
            if re.search(r'[HIhi]\(', candidate):
                chain.append(('', _norm_expr(candidate), ''))

    if len(chain) < 2:
        prose_chain = []
        for raw_line in raw_lines:
            line = raw_line.strip()
            if not line:
                continue
            line = re.sub(r'\$([^$]+)\$', r'\1', line)
            line = (line.replace('\u2264', '<=').replace('\u2265', '>=')
                        .replace('\u2a7d', '<=').replace('\u2a7e', '>='))
            line = re.sub(r'^(?:step\s*\d+[:.\s]*)', '', line, flags=re.IGNORECASE)
            line = re.sub(r'^\d+\.\s*', '', line)
            im = re.search(
                r'([^<>=]*[IHih]\([^)]+\)[^<>=]*)\s*([<>=]+)\s*([^<>=]*[IHih]\([^)]+\)[^<>=]*)',
                line
            )
            if im:
                lhs = im.group(1).strip().rstrip(':')
                rel = im.group(2).strip()
                rhs = im.group(3).strip()
                remainder = line[im.end():]
                reason_m = re.search(r'\((?:since\s+|by\s+)?(.+?)\)', remainder, re.IGNORECASE)
                if not reason_m:
                    reason_m = re.search(r'\bby\s+(.+?)(?:\.|$)', remainder, re.IGNORECASE)
                if not reason_m:
                    reason_m = re.search(r'\bsince\s+(.+?)(?:\.|$)', remainder, re.IGNORECASE)
                reason = reason_m.group(1).strip() if reason_m else ''
                if not prose_chain:
                    lhs_clean = re.sub(r'^.*?(?=[IHih]\()', '', lhs)
                    if re.search(r'[IHih]\(', lhs_clean):
                        prose_chain.append(('', _norm_expr(lhs_clean), ''))
                rhs_clean = re.sub(r'\s*\((?:since|by)\b.*$', '', rhs, flags=re.IGNORECASE).strip()
                prose_chain.append((rel, _norm_expr(rhs_clean), reason.lower()))
        if len(prose_chain) > len(chain):
            chain = prose_chain

    return [(r, e, rsn) for r, e, rsn in chain if len(e) >= 4]

# ══════════════════════════════════════════════════════════════
# PSITIP step-by-step proof verification
# ══════════════════════════════════════════════════════════════

_MARKOV_SEP_RE = re.compile(r'\s*[─\-]\s*')


def _rv(name):
    return globals().get(name.strip().upper())


def _parse_assumption_to_psitip(assumption):
    if not assumption or assumption.strip().lower() in ('none', '(none)', ''):
        return None
    constraints = []
    for part in re.split(r'\s*&\s*', assumption):
        part = part.strip()
        if not part:
            continue
        tokens = _MARKOV_SEP_RE.split(part)
        if (len(tokens) >= 3
                and all(len(t.strip()) == 1 and t.strip().isalpha() for t in tokens)):
            rvs = [_rv(t) for t in tokens]
            if all(v is not None for v in rvs):
                try:
                    constraints.append(markov(*rvs))
                    continue
                except Exception:
                    pass
        m = re.match(r'^indep\(\s*([A-Za-z])\s*,\s*([A-Za-z])\s*\)$', part)
        if m:
            v1, v2 = _rv(m.group(1)), _rv(m.group(2))
            if v1 is not None and v2 is not None:
                try:
                    constraints.append(indep(v1, v2))
                    continue
                except Exception:
                    pass
        # Handle H(A|B) = 0 / H(A|B) == 0  (Y is a deterministic function of X).
        # Also handles H(A) = 0 (constant RV) and I(A;B) = 0 (independence).
        he_m = re.match(
            r'^([HIhi])\(\s*([A-Za-z,+]+)\s*(?:\|\s*([A-Za-z,+]+))?\s*\)\s*==?\s*0\s*$',
            part)
        if he_m:
            fn_char = he_m.group(1).upper()   # H or I
            args    = he_m.group(2).strip()
            cond    = (he_m.group(3) or '').strip()
            expr_s  = (f"{fn_char}({args}|{cond})" if cond else f"{fn_char}({args})")
            try:
                pe_h = Expr.parse(expr_s)
                constraints.append(pe_h == 0)
                continue
            except Exception:
                pass
    if not constraints:
        return None
    result = constraints[0]
    for c in constraints[1:]:
        try:
            result = result & c
        except Exception:
            pass
    return result


def _strip_trailing_annotation(s):
    s = s.rstrip()
    if not s.endswith(')'):
        return s
    depth = 0
    for j in range(len(s) - 1, -1, -1):
        if s[j] == ')':
            depth += 1
        elif s[j] == '(':
            depth -= 1
            if depth == 0:
                before = s[:j].rstrip()
                if before and before[-1].isalpha():
                    return s
                return before
    return s


def _restore_case_for_parse(expr_norm):
    result = re.sub(r'(?<![a-zA-Z])h\(', 'H(', expr_norm)
    result = re.sub(r'(?<![a-zA-Z])i\(', 'I(', result)
    result = re.sub(r'\b([a-z])\b', lambda m: m.group(1).upper(), result)
    return result


_LEADING_ANNOT_RE = re.compile(r'^\([^()]{1,40}\)\s*')


def _strip_inline_annotations(s: str) -> str:
    result = []
    i = 0
    while i < len(s):
        result.append(s[i])
        if s[i] == ')' and i + 1 < len(s):
            j = i + 1
            while j < len(s) and s[j] == ' ':
                j += 1
            if j < len(s) and s[j] == '(':
                depth = 0
                k = j
                while k < len(s):
                    if s[k] == '(':
                        depth += 1
                    elif s[k] == ')':
                        depth -= 1
                        if depth == 0:
                            break
                    k += 1
                content = s[j + 1:k].strip()
                is_annotation = (
                    bool(content)
                    and not re.match(r'^\d', content)
                    and not re.match(r'^[IiHh]\s*\(', content)
                )
                if is_annotation:
                    i = k + 1
                    continue
        i += 1
    return ''.join(result)

def _expand_co_info(expr_str: str) -> str:
    def _sub(m):
        coef = m.group(1) or ''
        a    = m.group(2).strip()
        b    = m.group(3).strip()
        rest = m.group(4).strip()
        if '|' in rest:
            c, d = rest.split('|', 1)
            c, d = c.strip(), d.strip()
            return f"{coef}I({a};{b}|{d})-{coef}I({a};{b}|{c},{d})"
        return f"{coef}I({a};{b})-{coef}I({a};{b}|{rest})"
    pat = re.compile(r'(\d+\*?)?[Ii]\(([^;)]+);([^;)|]+);([^)]+)\)', re.IGNORECASE)
    return pat.sub(_sub, expr_str)


_CV_ALIAS_POOL = 'UV'


def _build_copy_var_context(proof_text: str, base_constraints=None):
    if not proof_text:
        return {}, base_constraints
    cv_tokens = set(m.group(0) for m in _COPY_VAR_RE.finditer(proof_text))
    if not cv_tokens:
        return {}, base_constraints
    base_vars: set = set()
    for bm in re.finditer(r'[IiHh]\(([^)]+)\)', proof_text):
        for tok in re.split(r'[;,|()+\-* \t]+', bm.group(1)):
            tok = tok.strip()
            if (len(tok) == 1 and tok.isalpha()
                    and tok.upper() not in ('I', 'H')
                    and not _COPY_VAR_RE.search(tok)):
                base_vars.add(tok.upper())
    in_use = base_vars.copy()
    alias_map: dict = {}
    for cv in sorted(cv_tokens):
        for cand in _CV_ALIAS_POOL:
            if cand not in in_use:
                alias_map[cv] = cand
                in_use.add(cand)
                break
    if not alias_map:
        return {}, base_constraints
    base_comp = None
    for v in sorted(base_vars):
        comp = _rv(v)
        if comp is not None:
            base_comp = comp if base_comp is None else (base_comp + comp)
    constraints = base_constraints
    if base_comp is not None:
        for alias in alias_map.values():
            alias_comp = _rv(alias)
            if alias_comp is None:
                continue
            try:
                ind = indep(alias_comp, base_comp)
                constraints = ind if constraints is None else (constraints & ind)
            except Exception:
                pass
    return alias_map, constraints


def _apply_cv_alias(expr_str: str, alias_map: dict) -> str:
    if not alias_map:
        return expr_str
    for orig, alias in sorted(alias_map.items(), key=lambda x: -len(x[0])):
        if any(c in orig for c in ("'", "\u2019", "\u02bc")):
            expr_str = expr_str.replace(orig, alias)
        else:
            expr_str = re.sub(
                r'(?<![A-Za-z0-9])' + re.escape(orig) + r'(?![A-Za-z0-9_])',
                alias, expr_str,
            )
    return expr_str

def _preprocess_expr_for_parse(s: str) -> str:
    """Fix common llm expression-formatting issues before Expr.parse().

    Handles:
      - Trailing 'so:...' tail       : 'I(X;Y),so:' -> 'I(X;Y)'
      - Square bracket grouping      : '[I(X;Y)]'   -> '(I(X;Y))'
      - Trailing prose after )       : ')theword'   -> ')'
      - Inline 'by...' annotation    : ')by CR...'  -> ')'
      - Top-level comma list         : 'I(X;Y),I(Z)' -> 'I(X;Y)'
      - Missing multiplication sign  : '2H(X)'      -> '2*H(X)'
      - I() with comma not semicolon : 'I(X,Y|Z)'   -> 'I(X;Y|Z)'
      - H() with semicolons          : 'H(X;Y)'     -> 'H(X,Y)'
    """
    # Strip trailing 'so:...' conclusion tail (e.g. 'I(X;Y),so:I(Z)<=...' -> 'I(X;Y)')
    s = re.sub(r'\s*,?\s*so\s*:.*$', '', s, flags=re.IGNORECASE).strip()
    # Replace square-bracket grouping with parens (e.g. '[I(X;Y)]' -> '(I(X;Y))')
    s = s.replace('[', '(').replace(']', ')')
    # Strip trailing prose after last closing paren (e.g. ')thebracket<=0' -> ')')
    s = re.sub(r'\)\s*[A-Za-z][^()]*$', ')', s).strip()
    # Strip 'by...' inline annotation after closing paren (e.g. ')by chain rule...' -> ')')
    s = re.sub(r'(?<=\))\s*by\b.*$', '', s, flags=re.IGNORECASE).strip()
    # Truncate at first top-level comma (e.g. 'I(X;Y),I(Z;W)' -> 'I(X;Y)')
    _depth = 0
    for _i, _ch in enumerate(s):
        if _ch == '(':
            _depth += 1
        elif _ch == ')':
            _depth -= 1
        elif _ch == ',' and _depth == 0:
            s = s[:_i].strip()
            break
    # 2H( -> 2*H(  and  3I( -> 3*I(
    s = re.sub(r'(\d)\s*([HhIi])\s*\(', r'\1*\2(', s)
    # I(X,Y|Z) -> I(X;Y|Z) : single comma before '|' with no existing semicolon
    def _fix_i_comma(m):
        inner = m.group(1)
        if ';' not in inner and ',' in inner:
            if '|' in inner:
                pre, post = inner.split('|', 1)
                if pre.count(',') == 1:
                    return 'I(' + pre.replace(',', ';', 1) + '|' + post + ')'
            elif inner.count(',') == 1:
                return 'I(' + inner.replace(',', ';', 1) + ')'
        return m.group(0)
    s = re.sub(r'[Ii]\(([^)]+)\)', _fix_i_comma, s)
    # H(X;Y) -> H(X,Y) : semicolons in H() without '|' are mis-typed commas
    def _fix_h_semi(m):
        inner = m.group(1)
        if ';' in inner and '|' not in inner:
            return 'H(' + inner.replace(';', ',') + ')'
        return m.group(0)
    s = re.sub(r'[Hh]\(([^)]+)\)', _fix_h_semi, s)
    return s


def _safe_parse_expr(expr_str: str):
    """Parse expr_str to an Expr with multi-stage cleanup.

    Stage 0: as-is
    Stage 1: strip inline annotations
    Stage 2: strip leading annotation + truncate at '='
    Stage 3: both 1+2
    Stage 4: expression formatting normalization (missing *, wrong separators)
    """
    try:
        return Expr.parse(expr_str)
    except Exception:
        pass
    s1 = _strip_inline_annotations(expr_str)
    if s1 != expr_str:
        try:
            return Expr.parse(s1)
        except Exception:
            pass
    else:
        s1 = expr_str
    s2 = _LEADING_ANNOT_RE.sub('', expr_str).strip()
    if '=' in s2:
        s2 = s2.split('=', 1)[0].strip()
    if s2 != expr_str:
        try:
            return Expr.parse(s2)
        except Exception:
            pass
    s3 = _LEADING_ANNOT_RE.sub('', s1).strip()
    if '=' in s3:
        s3 = s3.split('=', 1)[0].strip()
    try:
        return Expr.parse(s3)
    except Exception:
        pass
    # Stage 4: expression formatting normalization
    s4 = _preprocess_expr_for_parse(expr_str)
    if s4 != expr_str:
        try:
            return Expr.parse(s4)
        except Exception:
            pass
        s41 = _strip_inline_annotations(s4)
        if s41 != s4:
            try:
                return Expr.parse(s41)
            except Exception:
                pass
        s42 = _preprocess_expr_for_parse(s1)
        if s42 != s1:
            try:
                return Expr.parse(s42)
            except Exception:
                pass
    # Final raise (caught by caller)
    return Expr.parse(s3 if s3 else expr_str)


def _extract_axioms(proof_text: str) -> set:
    """Return set of axiom-type strings cited in proof_text (broadened patterns)."""
    axioms = set()
    for m in re.finditer(r'\(since\b([^\n]*)', proof_text, re.IGNORECASE):
        r_text = m.group(1).lower().strip().rstrip(')')
        if 'copy lemma' in r_text:
            axioms.add('copy_lemma')
            continue
        if _has_copy_vars(m.group(1)):
            continue
        if 'markov'                              in r_text: axioms.add('markov')
        if 'dpi' in r_text or 'data processing' in r_text: axioms.add('dpi')
        if 'chain rule'                          in r_text: axioms.add('chain_rule')
        if 'indep'                               in r_text: axioms.add('independence')
        if 'conditioning'                        in r_text: axioms.add('cond_reduces')
        if re.search(r'h\([^|)]+\)\s*>=\s*0',   r_text): axioms.add('H>=0')
        if re.search(r'h\([^)]+\|[^)]+\)\s*>=\s*0', r_text): axioms.add('H|>=0')
        if re.search(r'i\([^;)]+;[^)|]+\)\s*>=\s*0', r_text): axioms.add('I>=0')
        if re.search(r'i\([^;)]+;[^)]*\|[^)]+\)\s*>=\s*0', r_text): axioms.add('I|>=0')
        if re.search(r'h\([^)]+\)\s*=+\s*0',    r_text): axioms.add('H=0')
    tl = proof_text.lower()
    if 'copy lemma' in tl: axioms.add('copy_lemma')
    if not _has_copy_vars(proof_text) and re.search(r'\bmarkov\b', tl): axioms.add('markov')
    if re.search(r'=\s*0\b', tl): axioms.add('H=0')
    if re.search(r'\b(?:data\s+processing(?:\s+inequality)?|dpi)\b', tl): axioms.add('dpi')
    if re.search(r'\bchain\s+rule\b', tl): axioms.add('chain_rule')
    if re.search(r'\bmarkov\b', tl): axioms.add('markov')
    if re.search(r'\bsubmodular', tl): axioms.add('cond_reduces')
    if re.search(r'\bnon-?negativ', tl):
        if re.search(r'i\([^)]+\)\s*>=\s*0', tl): axioms.add('I>=0')
        elif re.search(r'h\([^)]+\|[^)]+\)\s*>=\s*0', tl): axioms.add('H|>=0')
        else: axioms.add('I>=0')
    if re.search(r'\bindepend', tl) and not _has_copy_vars(proof_text): axioms.add('independence')
    if re.search(r'i\([^)]+\)\s*>=?\s*0', tl): axioms.add('I>=0')
    if re.search(r'i\([^)]+\|[^)]+\)\s*>=?\s*0', tl): axioms.add('I|>=0')
    if re.search(r'h\([^)]+\)\s*>=?\s*0', tl): axioms.add('H>=0')
    if re.search(r'h\([^)]+\|[^)]+\)\s*>=?\s*0', tl): axioms.add('H|>=0')
    if re.search(r'(?:copies|copy\s+lemma|independent\s+cop)', tl): axioms.add('copy_lemma')
    return axioms


_NS_CITE_RE = re.compile(
    r'\b(?:zhang|zheng|non.?shannon|copy\s+lemma|shenoy|connected|ahlswede'
    r'|matus|ingleton|non.?linear)',
    re.IGNORECASE,
)


def verify_proof_steps_psitip(llm_proof, assumption=None, goal_expr_norm=None):
    """Verify each consecutive step via PSITIP LP.

    Failing step classification:
      [LP-contradicted]  : reversed direction holds -> step is definitely wrong
      [LP-unproven]      : neither direction holds AND proof is Shannon (no copy vars
                           AND no named non-Shannon reference)
      [non-Shannon step] : neither direction holds AND (proof uses copy variables
                           OR proof cites a non-Shannon result by name)
                           -> expected non-Shannon reasoning, excluded from score
    """
    chain = _parse_chain(llm_proof)
    if len(chain) < 2:
        return None, 0, 0, ["  (fewer than 2 steps -- nothing to verify)"]
    constraints = _parse_assumption_to_psitip(assumption) if assumption else None
    alias_map, copy_constraints = _build_copy_var_context(llm_proof, constraints)
    # Fix D+E: a proof uses non-Shannon reasoning if it has copy variables
    # OR if it explicitly cites a named non-Shannon result
    _proof_uses_cv   = len(alias_map) > 0
    _proof_cites_ns  = bool(_NS_CITE_RE.search(llm_proof))
    _proof_is_nonshan = _proof_uses_cv or _proof_cites_ns
    n_valid = 0
    n_lp_contradicted = 0
    n_lp_unproven = 0
    n_ns_step = 0
    n_total = len(chain) - 1
    details = [f"  Consecutive ({n_total} transitions)  assumption={assumption!r}"]
    n_parse_err = 0
    n_goal_skip = 0
    for i in range(1, len(chain)):
        rel, curr_norm, _ = chain[i]
        _, prev_norm, _ = chain[i-1]
        has_cv = _has_copy_vars(prev_norm) or _has_copy_vars(curr_norm)
        prev_str = _restore_case_for_parse(_strip_trailing_annotation(prev_norm))
        curr_str = _restore_case_for_parse(_strip_trailing_annotation(curr_norm))
        prev_str = _expand_co_info(_apply_cv_alias(prev_str, alias_map))
        curr_str = _expand_co_info(_apply_cv_alias(curr_str, alias_map))
        step_constr = copy_constraints if has_cv else constraints
        try:
            pe_prev = _safe_parse_expr(prev_str)
            pe_curr = _safe_parse_expr(curr_str)
        except Exception as ex:
            n_parse_err += 1
            n_total -= 1
            details.append(
                f"    {_SYM_DASH} parse_err [{i-1}->{i}]: "
                f"{prev_str[:20]} {rel} {curr_str[:20]}  ({ex})")
            continue
        step_tag = ""
        try:
            if '<=' in rel:
                step_region = (pe_prev <= pe_curr)
            elif '>=' in rel:
                step_region = (pe_prev >= pe_curr)
            else:
                step_region = (pe_prev == pe_curr)
            ok = bool(step_constr >> step_region) if step_constr is not None else bool(step_region)
            if not ok:
                if '<' not in rel and '>' not in rel:
                    try:
                        fwd = (bool(step_constr >> (pe_prev <= pe_curr))
                               if step_constr is not None else bool(pe_prev <= pe_curr))
                        rev = (bool(step_constr >> (pe_curr <= pe_prev))
                               if step_constr is not None else bool(pe_curr <= pe_prev))
                        if fwd and rev:
                            ok = True
                        elif not fwd and not rev:
                            if _proof_is_nonshan:
                                step_tag = " [non-Shannon step]"
                                n_ns_step += 1
                            else:
                                step_tag = " [LP-unproven]"
                                n_lp_unproven += 1
                        else:
                            step_tag = " [LP-contradicted]"
                            n_lp_contradicted += 1
                    except Exception:
                        step_tag = " [LP-unproven]"
                        n_lp_unproven += 1
                else:
                    try:
                        rev_region = (pe_curr <= pe_prev) if '<=' in rel else (pe_prev <= pe_curr)
                        rev_ok = (bool(step_constr >> rev_region)
                                  if step_constr is not None else bool(rev_region))
                        if rev_ok:
                            step_tag = " [LP-contradicted]"
                            n_lp_contradicted += 1
                        elif _proof_is_nonshan:
                            step_tag = " [non-Shannon step]"
                            n_ns_step += 1
                        else:
                            step_tag = " [LP-unproven]"
                            n_lp_unproven += 1
                    except Exception:
                        step_tag = " [LP-unproven]"
                        n_lp_unproven += 1
        except Exception as ex:
            details.append(f"    {_SYM_DASH} eval_err [{i-1}->{i}]: {ex}")
            n_total -= 1
            continue
        if not ok and goal_expr_norm is not None:
            if _sim(curr_norm, goal_expr_norm) >= 0.75:
                n_goal_skip += 1
                n_total -= 1
                details.append(f"    {_SYM_DASH} goal_skip [{i-1}->{i}]: conclusion re-stated")
                continue
        if "[non-Shannon step]" in step_tag:
            n_total -= 1
            details.append(
                f"    {_SYM_DASH} ns_step [{i-1}->{i}]  "
                f"{prev_str[:24]} {rel} {curr_str[:24]}{step_tag}")
            continue
        n_valid += int(ok)
        sym = _SYM_CHECK if ok else _SYM_CROSS
        tag = "" if ok else step_tag
        details.append(f"    {sym} [{i-1}->{i}]  {prev_str[:24]} {rel} {curr_str[:24]}{tag}")
    score = (n_valid / n_total) if n_total > 0 else None
    if n_goal_skip:
        details.append(f"  ({n_goal_skip} transition(s) skipped -- conclusion re-statement)")
    if n_parse_err:
        details.append(f"  ({n_parse_err} transition(s) skipped due to parse errors)")
    if n_ns_step:
        details.append(
            f"  ({n_ns_step} step(s) [non-Shannon step] -- copy-lemma/named inequality, excluded from score)")
    if n_lp_contradicted:
        details.append(f"  ({n_lp_contradicted} step(s) LP-contradicted)")
    if n_lp_unproven:
        details.append(f"  ({n_lp_unproven} step(s) LP-unproven)")
    return score, n_valid, n_total, details


def compare_proofs(psitip_proof, psitip_result, psitip_type,
                   llm_proof, llm_verdict, _llm_type,
                   assumption=None):
    """Compare PSITIP vs llm proof chains.

    Components:
      psitip_consec : consecutive-step LP verification score
      axiom         : axiom recall (boosted when LP fully verified)
      endpoint_ok   : LP check that chain[0] <= chain[-1]
      lp_contradicted / lp_unproven : per-step failure counts
    """
    p_true = psitip_result if isinstance(psitip_result, bool) else (psitip_result == 'True')
    c_true = (llm_verdict == 'True')
    if p_true != c_true:
        return 0.0, "Verdict mismatch -- proofs not comparable.", {}
    if not p_true:
        return None, "Both False -- no proof to compare.", {}
    if not psitip_proof:
        return None, "PSITIP returned True but no proof text available.", {}
    if not llm_proof or llm_proof.startswith('(no proof'):
        return 0.0, "llm returned True but no proof chain extracted.", {}
    details = []
    p_chain = _parse_chain(psitip_proof)
    c_chain = _parse_chain(llm_proof)
    if not p_chain:
        return None, "Could not parse PSITIP proof chain.", {}
    if not c_chain and not _has_copy_vars(llm_proof):
        return 0.0, "Could not parse llm proof chain.", {}
    details.append(f"  PSITIP steps: {len(p_chain)}   llm steps: {len(c_chain)}")

    # ── Component 1: Axiom recall ─────────────────────────────────────────────
    p_ax = _extract_axioms(psitip_proof)
    c_ax = _extract_axioms(llm_proof)
    if p_ax:
        compat = sum(1 for a in p_ax if c_ax & _AXIOM_COMPAT.get(a, {a}))
        ax_recall = compat / len(p_ax)
    else:
        ax_recall = 1.0
    if ax_recall == 0.0 and len(c_chain) >= 2:
        ax_recall = 0.5
    details.append(f"  Axiom recall: {ax_recall:.0%}  (PSITIP={p_ax or 'none'}, llm={c_ax or 'none'})")

    # ── Component 2: PSITIP LP consecutive verification ───────────────────────
    _psitip_goal_norm = next(
        (e for _, e, _ in reversed(p_chain) if not _has_copy_vars(e)), None
    )
    pv_score, pv_n_valid, pv_n_total, pv_details = verify_proof_steps_psitip(
        llm_proof, assumption=assumption, goal_expr_norm=_psitip_goal_norm
    )
    if pv_score is not None:
        details.append(f"  PSITIP verify: consec={pv_n_valid}/{pv_n_total}  => {pv_score:.0%}")
    else:
        details.append(f"  PSITIP verify: N/A (no standard-variable steps found)")
    details.extend(pv_details)

    # ── Component 3: LP endpoint check ───────────────────────────────────────
    _endpoint_ok = None
    if len(c_chain) >= 2 and pv_score is not None and not _has_copy_vars(llm_proof):
        try:
            _ep_constr = _parse_assumption_to_psitip(assumption) if assumption else None
            _pe_start = _safe_parse_expr(
                _restore_case_for_parse(_strip_trailing_annotation(c_chain[0][1])))
            _pe_end = _safe_parse_expr(
                _restore_case_for_parse(_strip_trailing_annotation(c_chain[-1][1])))
            _ep_region = (_pe_start <= _pe_end)
            _endpoint_ok = (bool(_ep_constr >> _ep_region)
                            if _ep_constr is not None else bool(_ep_region))
            details.append(
                f"  Endpoint (chain[0]<=chain[-1]): "
                f"{'✓ LP-proved' if _endpoint_ok else '✗ LP-fails'}")
        except Exception:
            _endpoint_ok = None

    # ── LP-verified axiom boost ───────────────────────────────────────────────
    if pv_score is not None:
        if pv_score >= 1.0:
            if ax_recall < 1.0:
                details.append(f"  Axiom boost: {ax_recall:.0%} -> 100% (LP-verified)")
            ax_recall = 1.0
        elif pv_score >= 0.75:
            boosted = max(ax_recall, 0.75)
            if boosted > ax_recall:
                details.append(f"  Axiom boost: {ax_recall:.0%} -> {boosted:.0%} (LP >=75%)")
            ax_recall = boosted

    # ── Final score ───────────────────────────────────────────────────────────
    if pv_score is not None:
        score = _W_PSITIP_CONSEC * pv_score + _W_AXIOM * ax_recall
        pv_label = f"{pv_score:.0%}"
    else:
        score = ax_recall
        pv_label = "N/A"
    details.append(f"  {_SYM_DASH*2} Proof similarity : {score:.0%} {_SYM_DASH*2}")
    details.append(f"     pv={pv_label} axiom={ax_recall:.0%}")

    n_contradicted = sum(
        1 for d in pv_details if '[LP-contradicted]' in d and d.strip().startswith(_SYM_CROSS)
    )
    n_lp_unproven = sum(
        1 for d in pv_details if '[LP-unproven]' in d and d.strip().startswith(_SYM_CROSS)
    )
    components = {
        'psitip_consec':   pv_score,
        'axiom':           ax_recall,
        'lp_contradicted': n_contradicted if n_contradicted > 0 else None,
        'lp_unproven':     n_lp_unproven  if n_lp_unproven  > 0 else None,
        'endpoint_ok':     _endpoint_ok,
    }
    return score, '\n'.join(details), components


compare_proofs._lp_boost = True

print("Cell 3 loaded:")
print("  A. _preprocess_expr_for_parse: fixes 2H(), I(X,Y|Z), H(X;Y) formatting")
print("  B. compare_proofs: LP endpoint check (chain[0] <= chain[-1])")
print("  C. _parse_assumption_to_psitip: H(A|B)=0 / H(A|B)==0 parsing")
print("  D. verify_proof_steps_psitip: [non-Shannon step] for copy-var proofs")
print("  E. verify_proof_steps_psitip: [non-Shannon step] for named citations")


Cell 3 loaded:
  A. _preprocess_expr_for_parse: fixes 2H(), I(X,Y|Z), H(X;Y) formatting
  B. compare_proofs: LP endpoint check (chain[0] <= chain[-1])
  C. _parse_assumption_to_psitip: H(A|B)=0 / H(A|B)==0 parsing
  D. verify_proof_steps_psitip: [non-Shannon step] for copy-var proofs
  E. verify_proof_steps_psitip: [non-Shannon step] for named citations


In [10]:
# ── Cell 3.1: Parser patch for markdown-styled verdict lines ─────────────────
def parse_llm_response(text, out_tok=None, max_tok=3000):
    """Extract VERDICT, TYPE, and Proof section from llm response.

    Patch: tolerate markdown emphasis around labels, e.g.
      **VERDICT:** True   or   __VERDICT__: False
    """
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'```[\s\S]*?```', '\n', text)
    text = re.sub(r'`[^`]*`', '', text)

    verdict = "Unknown"
    verdict_labels = [r'VERDICT', r'VERDIC', r'RESULT', r'ANSWER', r'CONCLUSION']
    _labeled = []

    # Allow optional markdown wrappers around label and colon.
    # Examples matched:
    #   VERDICT: True
    #   **VERDICT:** True
    #   __VERDICT__: False
    for lbl in verdict_labels:
        pattern = (
            rf'^\s*(?:\*\*|__)?\s*{lbl}\s*(?:\*\*|__)?\s*[:\-]\s*([^\n]+)'
        )
        for m in re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE):
            cand = m.group(1).strip()
            v_m = re.search(r'\b(True|False|Yes|No)\b', cand, re.IGNORECASE)
            if v_m:
                v = v_m.group(1).lower()
                v_str = 'True' if v in ('true', 'yes') else 'False' if v in ('false', 'no') else None
                if v_str:
                    _labeled.append((m.start(), v_str))
    if _labeled:
        verdict = sorted(_labeled, key=lambda x: x[0])[-1][1]

    if verdict == 'Unknown':
        _concluding = []
        for m in re.finditer(r'(?:therefore|thus|conclusion|hence)[:,\s]*([^\n]+)', text, re.IGNORECASE):
            v_m = re.search(r'\b(True|False|Yes|No)\b', m.group(1), re.IGNORECASE)
            if v_m:
                v = v_m.group(1).lower()
                v_str = 'True' if v in ('true', 'yes') else 'False' if v in ('false', 'no') else None
                if v_str:
                    _concluding.append((m.start(), v_str))
        if _concluding:
            verdict = sorted(_concluding, key=lambda x: x[0])[-1][1]

    if verdict == 'Unknown':
        all_tf = re.findall(r'\b(True|False)\b', text[:300], re.IGNORECASE)
        if all_tf:
            verdict = all_tf[-1].capitalize()

    ptype = 'Unknown'
    for m in re.finditer(r'^\s*(?:\*\*|__)?\s*TYPE\s*(?:\*\*|__)?\s*[:\-]\s*(.+)$', text, re.IGNORECASE | re.MULTILINE):
        t = re.sub(r'^\*+\s*', '', m.group(1).strip())
        if t:
            ptype = t

    proof = None
    proof_starts = list(re.finditer(r'\bProof[:\s]*\n', text, re.IGNORECASE))
    if proof_starts:
        body = text[proof_starts[-1].end():]
        stop = re.search(r'\n(?:VERDICT|RESULT|ANSWER|CONCLUSION|END)[:\-\s]', body, re.IGNORECASE)
        proof = body[:stop.start()].strip() if stop else body.strip()
        proof = re.sub(
            r'^\s*VERDICT[:\-\s]*\S.*\n(?:TYPE[:\-\s]*.*\n)?(?:Proof[:\s]*\n)?',
            '', proof, flags=re.IGNORECASE
        ).strip()

    if not proof:
        proof = text.strip() if len(text.strip()) < 200 else None

    if out_tok is not None and out_tok >= max_tok:
        verdict = 'Unknown'
        return verdict, ptype, proof

    if proof and _ABANDON_RE.search(proof):
        _last_ab = list(_ABANDON_RE.finditer(proof))[-1]
        _line_end = proof.find('\n', _last_ab.end())
        if _line_end != -1:
            _remainder = proof[_line_end:].strip()
            if len(_remainder) >= 30:
                proof = _remainder
            elif verdict == 'True':
                verdict = 'Unknown'
        elif verdict == 'True':
            verdict = 'Unknown'

    return verdict, ptype, proof

print("Parser patch loaded: markdown-styled VERDICT/TYPE lines are supported")

Parser patch loaded: markdown-styled VERDICT/TYPE lines are supported


In [11]:
# ── Cell 4: System prompt (SYSTEM_PROMPT_VERDICT) ───────────────────────────────────
# ── PROMPTS ─────────────────────────────────────────────
#
#  SYSTEM_PROMPT_VERDICT  → Verdict + Type + Proof
#
#  True  cases → VERDICT + TYPE + step-by-step Proof
#  False cases → VERDICT + TYPE: N/A + brief counterexample / reasoning

# ── Verdict + Type + Proof ─────────────────────────────────────────────────

SYSTEM_PROMPT_VERDICT = """\
You are an expert in Shannon information theory.
Given an information inequality (or equality) and its constraints,
decide whether it is True or False, classify its type, and provide a proof
(or counterexample for False).

OUTPUT FORMAT:

For TRUE inequalities,  write:
VERDICT: True
TYPE: Shannon | Non-Shannon
Proof:
  [step-by-step derivation,  see rules below]

For FALSE inequalities,  write:
VERDICT: False
TYPE: N/A
Disproof:
  [brief counterexample or reasoning why it fails]

PROOF FORMAT (STRICT,  follow exactly):
  Write proofs as an algebraic chain, one expression per line:
    <starting expression>
    <= <next expression>    (<reason>)
    =  <next expression>    (<reason>)
    >= <next expression>    (<reason>)
  Each continuation line MUST start with <=, =, or >= followed by the expression.
  Put the justification in parentheses at the end: (chain rule), (since I(X;Y|Z) >= 0), etc.
  Do NOT use "Step 1:", "Step 2:", numbered lists, or prose paragraphs.
  Do NOT use LaTeX/markdown formatting ($...$, **bold**, etc.).
  See the examples below,  match their format exactly.

DEFINITIONS:
  True  = holds for ALL distributions satisfying the constraints.
  False = there exists a violating distribution satisfying the constraints.

TYPE DEFINITIONS (for True inequalities only):
  Shannon      = provable from elemental Shannon inequalities only
                 (chain rule, non-negativity, DPI, submodularity, Markov conditions).
                 Double Markov cases (X─Y─Z & Y─X─Z) are TYPE: Shannon.
  Non-Shannon  = requires the Copy Lemma.  Key signal:
                 4+ unconstrained variables, coefficients ≥ 2 on MI terms.

PROOF RULES:
  Shannon proofs:
    1. Start from the SMALLER (LHS) side of the inequality, always begin with the left-hand side.
    2. Each step must cite a reason: chain rule, non-negativity of I(·;·)/H(·|·),
       data processing inequality (DPI), submodularity, or Markov condition.
    3. The proof must be a continuous chain: each line transforms the previous line, with no skipped steps or unexplained jumps.
    4. WARNING: Co-information I(X;Y;Z) can be NEGATIVE, never assume I(X;Y;Z)>=0.
    5. CRITICAL : ONE AXIOM PER STEP — STRICT SEPARATION OF = AND <=/>= :
       Each line of the proof must apply exactly ONE elementary axiom.
       RULE A — = steps: use ONLY for exact algebraic identities that preserve the value
                (chain rule expansion, substituting a known-zero term, rearranging sums).
                No bound may be applied on an = step.
       RULE B — <=/>=  steps: use ONLY to apply a single bound (non-negativity, DPI,
                submodularity, or Markov). The two sides must differ ONLY in that one
                term is dropped or bounded — no simultaneous algebraic rearrangement.
       REARRANGE FIRST, THEN BOUND:
         If you need to rearrange an expression AND apply a bound, you MUST do it in
         two separate lines: first an = step (rearrange), then a <=/>=  step (bound).
       Wrong — rearrangement and bound combined into one <= step:
         H(X,Y) + H(Y,Z)
         <= H(Y) + H(X,Y,Z)               (chain rule + submodularity in one line)
       Correct — rearrange first (=), then bound (>=):
         H(X,Y) + H(Y,Z)
         = H(Y) + H(X|Y) + H(Y,Z)         (chain rule on H(X,Y))
         = H(Y) + H(X|Y) + H(Y) + H(Z|Y) (chain rule on H(Y,Z))
         >= H(Y) + H(X,Z|Y) + H(Y)        (submodularity: H(X|Y)+H(Z|Y)>=H(X,Z|Y))
         = H(Y) + H(X,Y,Z)                (chain rule: H(Y)+H(X,Z|Y)=H(X,Y,Z))

  Non-Shannon (Copy Lemma) proofs:
    Step 1: Apply Copy Lemma,  state WHICH variable you copy and conditioned on WHAT.
            For variables (A,B,C,D), there exists D' s.t.: (D',C)=d(D,C), D'⊥⊥(A,B)|C.
    Step 2: State copy identities.
    Step 3: Chain of inequalities in the extended variable space.
    CRITICAL: EVERY chain expression that involves the copy variable MUST include D'
    explicitly in the expression.  Write I(A;B,D') not I(A;B+something).  A chain
    step may omit D' only if it is a valid Shannon inequality in the original space.

  NOTATION GUIDE:
  A─B─C     = Markov chain: I(A;C|B) = 0.
  A─B─C─D   = 4-variable Markov chain; DPI gives I(A;D)≤I(A;C)≤I(A;B).
  A & B      = both constraints hold simultaneously.
  indep(A,B) = A and B are marginally independent: I(A;B) = 0.
  H(A|B)==0  = A is a deterministic function of B.
  I(A&B)     = I(A;B)  (& separates the two arguments).
  A >> B     = "under constraint A, expression B holds".

══════════════════════════════════════════════════════════════════════════
IMPORTANT,  BE SKEPTICAL:
  Many inequalities that LOOK plausible are actually False.
  If you cannot construct a rigorous step-by-step proof from known
  identities and Shannon inequalities, try to construct a counterexample instead.
══════════════════════════════════════════════════════════════════════════

━━━ True,  Shannon ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

INEQUALITY : H(X,Y)+H(Y,Z) >= H(Y)+H(X,Y,Z)
CONSTRAINTS: (none)
VERDICT: True
TYPE: Shannon
Proof:
  H(X,Y) + H(Y,Z)
  = H(Y) + H(X|Y) + H(Y) + H(Z|Y)                (chain rule on both terms)
  = H(Y) + H(X|Y) + H(Z|Y) + H(Y)                (rearrange)
  >= H(Y) + H(X,Z|Y) + H(Y)                       (submodularity: H(X|Y)+H(Z|Y)>=H(X,Z|Y))
  = H(Y) + H(X,Y,Z)                               (chain rule: H(Y)+H(X,Z|Y)=H(X,Y,Z))

INEQUALITY : I(X;Z) <= I(X;Y)
CONSTRAINTS: X─Y─Z
VERDICT: True
TYPE: Shannon
Proof:
  I(X;Z)
  = I(X;Y) - I(X;Y|Z) + I(X;Z|Y)     (chain rule: I(X;Y,Z) expanded two ways)
  = I(X;Y) - I(X;Y|Z)                 (since X─Y─Z ⟹ I(X;Z|Y) = 0)
  <= I(X;Y)                            (since I(X;Y|Z) >= 0)

━━━ True,  Shannon (Double Markov) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

INEQUALITY : I(X;Z|Y) == 0
CONSTRAINTS: X─Y─Z & Y─X─Z
VERDICT: True
TYPE: Shannon
Proof:
  The result follows directly from the first constraint alone; the second constraint Y─X─Z is not needed.

INEQUALITY : H(Z|X) == H(Z|Y)
CONSTRAINTS: X─Y─Z & Y─X─Z
VERDICT: True
TYPE: Shannon
Proof:
  H(Z|X)
  = H(Z|X,Y)-I(X;Z|Y)   (since markov(Z, X, Y) and markov(X, Y, Z))
  = H(Z|Y)   (since markov(X, Y, Z))

  H(Z|Y)
  = H(Z|X,Y)   (since markov(X, Y, Z))
  <= H(Z|X)

━━━ True,  Non-Shannon (Copy Lemma) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

INEQUALITY : 2I(A;B) <= I(C;D)+I(C;A,B)+3I(A;B|C)+I(A;B|D)
CONSTRAINTS: (none)
VERDICT: True
TYPE: Non-Shannon
Proof:
  This is the Zhang-Yeung (1998) inequality,  the first known non-Shannon
  inequality.

  Step 1: Apply the Copy Lemma to D given C, producing D':
    (D', C) =d (D, C),   D' ⊥⊥ (A, B) | C

  Step 2: Copy identities:
    (a) I(A; D') = I(A; D)             (same marginals via (D',C)=d(D,C))
    (b) I(A; D'|C) = I(A; D|C)        (same conditional given C)
    (c) I(D'; A, B | C) = 0           (independence from A,B given C)
    (d) H(D') = H(D), H(D'|C) = H(D|C)

  Step 3: Chain of inequalities in the extended space {A, B, C, D, D'}:

    2I(A; B)
    = 2I(A; B, D) - 2I(A; D|B)                     (chain rule)
    <= 2I(A; B, D) - 2I(A; D|B)
       + I(D; D'|A, B) + I(A; D'|C)                 (non-negativity: both >= 0)
    = 2I(A; B, D) - 2I(A; D|B)
       + I(D; D'|A, B) + I(A; D|C)                  (copy identity (b))
    <= 2I(A; B, D, D') - 2I(A; D|B)
       + I(D; D'|A, B) + I(A; D|C)                  (I(A;B,D) <= I(A;B,D,D'))
    = 2I(A; B, D') + 2I(A; D|B, D') - 2I(A; D|B)
       + I(D; D'|A, B) + I(A; D|C)                  (chain rule on I(A;B,D,D'))
    = 2I(A; B, D') - I(A; D|B) + I(A; D|B, D')
       + I(D; D'|A, B) + I(A; D|C)                  (collect -2+1=-1 on I(A;D|B))

  Now use submodularity: I(A;D|B,D') + I(D;D'|A,B)
    <= I(A;D,D'|B) = I(A;D|B) + I(A;D'|B,D)        (chain rule)
  So: I(A;D|B,D') + I(D;D'|A,B) - I(A;D|B) <= I(A;D'|B,D)

    2I(A; B)
    <= 2I(A; B, D') + I(A; D'|B, D) + I(A; D|C)
    = I(A; B, D') + I(A; B) + I(A; D'|B, D) + I(A; D|C)   (split 2 = 1 + 1)
    So: I(A; B) <= I(A; B, D') + I(A; D'|B, D) + I(A; D|C)
    = I(A; B, D, D') + I(A; D|C)                     (chain rule recombining)

  Now apply I(D';A,B|C)=0 (copy identity (c)):
    I(A; B, D, D') = I(A; B, D) + I(A; D'|B, D)
    and I(A; D'|B, D) <= I(A; D'|C) + I(C; B, D) - I(C; B, D|A)
    ... further applying submodularity and copy identities gives:

    I(A; B) <= I(A; B, D') + I(A; D'|B, D) + I(A; D|C)
    <= I(C; D) + I(C; A, B) + 3I(A; B|C) + I(A; B|D)    (after simplification)

  Rearranging: 2I(A;B) <= I(C;D) + I(C;A,B) + 3I(A;B|C) + I(A;B|D).

  All intermediate steps use only Shannon inequalities on the extended
  set {A, B, C, D, D'}.  The Copy Lemma is essential: this inequality
  has no Shannon-only proof on {A, B, C, D} alone (Zhang-Yeung 1998).

INEQUALITY : H(X|Y) >= H(X|Z,W)
CONSTRAINTS: X─Y─Z─W
VERDICT: False
TYPE: N/A
Disproof:
  The ≤ direction is true by DPI, but ≥ is False.
  Counter: X uniform bit, Y=X, Z=W=const → H(X|Y)=0, H(X|Z,W)=1.

INEQUALITY : I(X;Y|Z) == 0
CONSTRAINTS: X─Y─Z & Y─X─Z
VERDICT: False
TYPE: N/A
Disproof:
  WARNING,  Double Markov pitfall.  X─Y─Z gives I(X;Z|Y)=0.  Y─X─Z gives
  I(Y;Z|X)=0.  But I(X;Y|Z)=0 does NOT follow.
  Counter: X=Y, Z=const → I(X;Y|Z)=H(X)>0.

INEQUALITY : H(X,Y|Z) == H(X|Z)+H(Y|Z)
CONSTRAINTS: X─Y─Z & Y─X─Z
VERDICT: False
TYPE: N/A
Disproof:
  Equivalent to I(X;Y|Z)==0; does not follow from these Markov chains.
  Counter: X=Y uniform bit, Z=const → H(X,Y|Z)=1, but H(X|Z)+H(Y|Z)=2.

INEQUALITY : 2I(X;Y) <= 3I(X;Y|Z)+3I(X;Z|Y)+3I(Y;Z|X)
CONSTRAINTS: (none)
VERDICT: False
TYPE: N/A
Disproof:
  Only 3 variables,  no non-Shannon inequality can help.  Cannot be proved
  by Shannon arguments either,  not enough slack.  False.

"""
print("SYSTEM_PROMPT_VERDICT :", len(SYSTEM_PROMPT_VERDICT), "chars")


SYSTEM_PROMPT_VERDICT : 9722 chars


In [12]:
# ── Cell 5: Load test cases (inequality labels & PSITIP expressions) ────────
# ── Load inequality labels ────────────────────────────────────
INEQ_FILE = rf"{PROJECT_DIR}\all_ineq_ordered.txt"
with open(INEQ_FILE, encoding="utf-8") as f:
    all_labels = [line.strip() for line in f if line.strip()]

# ── Load PSITIP expressions from external module ─────────────
_spec = importlib.util.spec_from_file_location("psitip_exprs", rf"{PROJECT_DIR}\psitip_exprs.py")
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
psitip_exprs = _mod.psitip_exprs
CL  = _mod.CL

assert len(psitip_exprs) == len(all_labels), (
    f"psitip_exprs has {len(psitip_exprs)} entries but label file has {len(all_labels)}"
)

N_CASES = 75   # ← set to len(all_labels) to run all 75
labels  = all_labels[:N_CASES]

print(f"Loaded {len(all_labels)} inequalities  ·  {len(psitip_exprs)} PSITIP expressions")
print(f"Running first {N_CASES}:\n")
for i, lbl in enumerate(labels, 1):
    print(f"  #{i}: {lbl}")


def _parse_assumption(label):
    """Return assumption string (before  =>  ) or None if unconditional."""
    if "  =>  " in label:
        return label.split("  =>  ", 1)[0].strip()
    return None


def _ctx_to_type(ctx_fn):
    """Map ctx_fn to a human-readable proof type string."""
    if ctx_fn is None:
        return "Shannon"
    return "Non-Shannon"   # CL

test_cases = [
    (lbl, _parse_assumption(lbl), region, ctx_fn, _ctx_to_type(ctx_fn))
    for lbl, (region, ctx_fn) in zip(labels, psitip_exprs[:N_CASES])
]

Loaded 75 inequalities  ·  75 PSITIP expressions
Running first 75:

  #1: H(X)+H(Y)+H(Z) >= H(X,Y,Z)
  #2: H(X)+I(Y;Z|X) >= I(Y;Z)
  #3: H(X,Y) <= H(X)+H(Y)
  #4: X─Y─Z  =>  H(Y) >= I(X;Z)
  #5: H(Y|X)==0  =>  I(Y;Z) <= I(X;Z)
  #6: H(Y|X)==0  =>  H(Y,Z) <= H(X,Z)
  #7: H(Y|X)==0  =>  I(Y;Z|W) <= I(X;Z|W)
  #8: H(Y|X)==0 & H(Z|Y)==0  =>  I(Z;W) <= I(X;W)
  #9: H(Y|X)==0 & X─Y─Z  =>  H(Z|Y) >= H(Z|X)
  #10: X─Y─Z  =>  I(X;Y) >= I(X;Z)
  #11: X─Y─Z─W  =>  I(X;W) <= I(X;Y)
  #12: X─Y─Z─W  =>  I(X;W) <= I(X;Z)
  #13: X─Y─Z─W  =>  I(X;W) <= I(Y;W)
  #14: X─Y─Z─W  =>  I(X;Z)+I(X;W) <= 2I(X;Y)
  #15: X─Y─Z─W  =>  I(X;W) <= H(Y)
  #16: X─Y─Z & H(W|Y)==0  =>  I(X;W) <= I(X;Y)
  #17: (X,W)─Y─Z  =>  I(X;Z) <= I(X;Y)
  #18: (X,W)─Y─Z  =>  I(W;Z) <= I(W;Y)
  #19: X─Y─Z  =>  I(X;Y) >= I(X;Z)+I(X;Z|Y)
  #20: X─Y─Z─W  =>  2H(Y) >= I(X;W)+I(X;Z)
  #21: X─Y─Z & Y─X─Z  =>  I(X;Z|Y) == 0
  #22: X─Y─Z & Y─X─Z  =>  I(Y;Z|X) == 0
  #23: X─Y─Z & Y─X─Z  =>  I(X;Z) == I(Y;Z)
  #24: X─Y─Z & Y─X─Z  =>  H(Z|X) == 

In [13]:
# ── Cell 6: Run pipeline (batch or sequential API calls + PSITIP ground truth) ─
# ══════════════════════════════════════════════════════════════════════════════
#  RUN,  single-pass pipeline
#
#  SYSTEM_PROMPT_VERDICT  → Verdict + Type + Proof
#
#  USE_BATCH=True  → Anthropic Batch API (~50% cheaper)
#  USE_BATCH=False → Sequential real-time calls
#  FORCE_RERUN=True → ignore checkpoint and start fresh
# ══════════════════════════════════════════════════════════════════════════════

BATCH_RESULTS_FILE = pathlib.Path(RESULT_DIR) / f"batch_results_{MODEL}_{_ts}.json"

# ── Checkpoint helpers ────────────────────────────────────────────────────────
def _load_checkpoint(path):
    """Load from exact path; if not found, fall back to the most-recent matching file."""
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    # Fallback: any checkpoint for the same model, most recent first
    pattern = str(path.parent / f"batch_results_{MODEL}_*.json")
    candidates = sorted(glob.glob(pattern))
    if candidates:
        latest = candidates[-1]
        print(f"[checkpoint] Exact file not found; loading most-recent: {os.path.basename(latest)}")
        with open(latest, encoding="utf-8") as f:
            return json.load(f)
    return []


def _save_checkpoint(path, records):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


# ── Batch submission helper ───────────────────────────────────────────────────
def _submit_and_wait(client, batch_reqs, desc=""):
    """Submit a batch and poll until complete. Returns {custom_id: result}."""
    from anthropic import APITimeoutError as _APITimeoutError

    print(f"\n[BATCH] Submitting {len(batch_reqs)} requests [{desc}] ...")
    batch    = client.messages.batches.create(requests=batch_reqs)
    batch_id = batch.id
    print(f"  Batch ID: {batch_id}  |  status: {batch.processing_status}")
    while True:
        # Retry polling on transient network / connect timeouts (default connect=5 s).
        for _poll_attempt in range(5):
            try:
                status = client.messages.batches.retrieve(batch_id)
                break
            except _APITimeoutError as _e:
                if _poll_attempt == 4:
                    raise
                print(f"  [poll] APITimeoutError (attempt {_poll_attempt+1}/5), "
                      f"retrying in 15 s …")
                time.sleep(15)
        counts = status.request_counts
        print(f"  [{datetime.datetime.now().strftime('%H:%M:%S')}]  "
              f"processing={counts.processing}  succeeded={counts.succeeded}  "
              f"errored={counts.errored}  expired={counts.expired}")
        if status.processing_status == "ended":
            break
        time.sleep(POLL_INTERVAL)
    return {r.custom_id: r for r in client.messages.batches.results(batch_id)}


# ── Generic single-pass runner ────────────────────────────────────────────────
def _build_user_prompt(label, assumption):
    # label may be "assumption  =>  inequality" or just "inequality"
    if "  =>  " in label:
        ineq_part = label.split("  =>  ", 1)[1].strip()
    else:
        ineq_part = label
    assumption_line = assumption if assumption else "(none)"
    return f"INEQUALITY : {ineq_part}\nCONSTRAINTS: {assumption_line}"


def _run_pass(case_list, system_prompt, desc=""):
    """Run one pass (batch or sequential) over case_list.
    case_list = [(i, label, assumption), ...]
    Returns {i: {"raw": str, "in_tok": int, "out_tok": int}}.
    """
    if not case_list:
        return {}

    if USE_BATCH:
        # ── Batch path ────────────────────────────────────────────────────────
        reqs = []
        for i, label, assumption in case_list:
            reqs.append({
                "custom_id": f"case-{i:03d}",
                "params": {
                    "model":       MODEL,
                    "max_tokens":  BATCH_MAX_TOKS,
                    "temperature": TEMPERATURE,
                    "system":      system_prompt,
                    "messages":    [{"role": "user",
                                     "content": _build_user_prompt(label, assumption)}],
                },
            })
        raw_results = _submit_and_wait(_client, reqs, desc)
        out = {}
        for i, label, assumption in case_list:
            cid    = f"case-{i:03d}"
            result = raw_results.get(cid)
            if result and result.result.type == "succeeded":
                text  = result.result.message.content[0].text
                usage = result.result.message.usage
                out[i] = {"raw": text,
                           "in_tok":  usage.input_tokens,
                           "out_tok": usage.output_tokens}
            else:
                err = result.result.type if result else "missing"
                print(f"  ⚠ #{i} failed: {err}")
                out[i] = {"raw": f"[API error: {err}]", "in_tok": 0, "out_tok": 0}
        return out

    else:
        # ── Sequential path ───────────────────────────────────────────────────
        out = {}
        for idx, (i, label, assumption) in enumerate(case_list, 1):
            print(f"  [{idx}/{len(case_list)}] #{i} …", end=" ", flush=True)
            for attempt in range(3):
                try:
                    text, usage = call_llm(label, assumption, system_prompt)
                    out[i] = {"raw": text,
                               "in_tok":  usage.input_tokens,
                               "out_tok": usage.output_tokens}
                    print(f"done ({usage.output_tokens} tok)")
                    break
                except Exception as e:
                    if attempt == 2:
                        print(f"ERROR: {e}")
                        out[i] = {"raw": f"[API error: {e}]", "in_tok": 0, "out_tok": 0}
                    else:
                        time.sleep(5)
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

all_case_list = [(i, lbl, assum)
                 for i, (lbl, assum, region, ctx_fn, ptype) in enumerate(test_cases, 1)]
saved    = _load_checkpoint(BATCH_RESULTS_FILE) if not FORCE_RERUN else []
done_ids = {r["num"] for r in saved}
run_cases = [(i, lbl, assum) for (i, lbl, assum) in all_case_list if i not in done_ids]

if saved:
    print(f"[checkpoint] Loaded {len(saved)} previously saved records; "
          f"running {len(run_cases)} remaining.")

print(f"[PASS 1]  True/False verdict + Type  ({len(run_cases)} cases)"
      f"  [{'BATCH' if USE_BATCH else 'SEQ'}]  max_tokens={BATCH_MAX_TOKS}")
print(f"{'='*64}")

p1_raw     = _run_pass(run_cases, SYSTEM_PROMPT_VERDICT, "P1")
p1_verdict = {}    # {i: "True" / "False" / "Unknown"}
p1_type    = {}    # {i: "Shannon" / "Non-Shannon" / "N/A" / "Unknown"}
p1_proof   = {}    # {i: proof string}

_max_tok = BATCH_MAX_TOKS
for _i, _raw_d in p1_raw.items():
    _raw  = _raw_d.get("raw", "")
    _otok = _raw_d.get("out_tok", 0)
    _v, _t, _proof = parse_llm_response(_raw, out_tok=_otok, max_tok=_max_tok)
    p1_verdict[_i] = _v
    p1_type[_i]    = _t
    p1_proof[_i]   = _proof or ""

# ── Build new records ─────────────────────────────────────────────────────────
new_records = []
total_in = total_out = 0
for i, (lbl, assum, region, ctx_fn, ptype) in enumerate(test_cases, 1):
    if i in done_ids:
        continue

    raw_d       = p1_raw.get(i, {})
    total_in   += raw_d.get("in_tok", 0)
    total_out  += raw_d.get("out_tok", 0)

    llm_v     = p1_verdict.get(i, "Unknown")
    llm_t_raw = p1_type.get(i, "Unknown")
    llm_proof = p1_proof.get(i, "")

    # Normalise type string
    if llm_v == 'True':
        t_lo = llm_t_raw.lower().strip()
        if 'copy' in t_lo:
            ctype = 'Non-Shannon'
        elif 'double' in t_lo or 'markov' in t_lo:
            ctype = 'Shannon'
        elif 'non-shannon' in t_lo or 'nonshannon' in t_lo:
            ctype = 'Non-Shannon'
        elif 'shannon' in t_lo:
            ctype = 'Shannon'
        else:
            ctype = 'Unknown'
    else:
        ctype = 'N/A'

    # PSITIP ground-truth verification
    psitip_result, psitip_proof = run_psitip(region, ctx_fn)
    psitip_v = "True" if psitip_result else "False"

    match = (psitip_v == llm_v)

    # Proof similarity scoring
    score, details, comps = compare_proofs(
        psitip_proof, psitip_result, ptype,
        llm_proof, llm_v, ctype,
        assumption=assum,
    )

    marker = "✓" if match else "✗"
    print(f"  #{i:>2}  PSITIP={psitip_v:<5}  llm={llm_v:<7}  {marker}  {lbl[:50]}")

    new_records.append({
        "num":              i,
        "label":            lbl,
        "assumption":       assum,
        "psitip":           psitip_v,
        "ptype":            ptype,
        "psitip_proof":     psitip_proof,
        "llm":           llm_v,
        "ctype":            ctype,
        "llm_proof":     llm_proof,
        "match":            match,
        "proof_score":      score,
        "proof_components": comps,
        "raw":              raw_d.get("raw", ""),
        "in_tok":           raw_d.get("in_tok", 0),
        "out_tok":          raw_d.get("out_tok", 0),
    })

# ── Merge + save ──────────────────────────────────────────────────────────────
results_summary = saved + new_records
_save_checkpoint(BATCH_RESULTS_FILE, results_summary)

# ── Summary ───────────────────────────────────────────────────────────────────
n_match = sum(1 for r in results_summary if r["match"])
n_total = len(results_summary)
print(f"\n{'='*64}")
print(f"  {n_total} cases finished  ✓={n_match}  ✗={n_total - n_match}  "
      f"({100 * n_match / n_total:.1f}% match)")
print(f"  Tokens  in={total_in:,}  out={total_out:,}")
print(f"  Saved → {BATCH_RESULTS_FILE.name}")


[PASS 1]  True/False verdict + Type  (75 cases)  [BATCH]  max_tokens=10000

[BATCH] Submitting 75 requests [P1] ...
  Batch ID: msgbatch_01VmdYyRC3zwiYsdomLXLV1G  |  status: in_progress
  [14:00:28]  processing=75  succeeded=0  errored=0  expired=0
  [14:01:29]  processing=75  succeeded=0  errored=0  expired=0
  [14:02:29]  processing=75  succeeded=0  errored=0  expired=0
  [14:03:30]  processing=75  succeeded=0  errored=0  expired=0
  [14:04:31]  processing=75  succeeded=0  errored=0  expired=0
  [14:05:31]  processing=0  succeeded=75  errored=0  expired=0
  # 1  PSITIP=True   llm=True     ✓  H(X)+H(Y)+H(Z) >= H(X,Y,Z)
  # 2  PSITIP=True   llm=True     ✓  H(X)+I(Y;Z|X) >= I(Y;Z)
  # 3  PSITIP=True   llm=True     ✓  H(X,Y) <= H(X)+H(Y)
  # 4  PSITIP=True   llm=True     ✓  X─Y─Z  =>  H(Y) >= I(X;Z)
  # 5  PSITIP=True   llm=True     ✓  H(Y|X)==0  =>  I(Y;Z) <= I(X;Z)
  # 6  PSITIP=True   llm=False    ✗  H(Y|X)==0  =>  H(Y,Z) <= H(X,Z)
  # 7  PSITIP=True   llm=True     ✓  H(Y|X)==0  =>  I

In [14]:
# ── Cell 7: Re-parse,  apply updated parser to stored raw outputs ────────────
# ── Re-parse stored raw outputs with the UPDATED parse_llm_response ────────
# Run this cell after loading a checkpoint (or after the pipeline cell) to apply
# parser fixes without spending any API budget.
# Updates 'llm', 'ctype', 'llm_proof', 'match' in-place; prints a diff.

_max_tok = globals().get('BATCH_MAX_TOKS', 8192)
print(f"Re-parsing {len(results_summary)} records  (max_tok={_max_tok}) ...\n")

_changed = []
for r in results_summary:
    old_v  = r.get('llm', 'Unknown')
    raw    = r.get('raw', '')
    out_tok = r.get('out_tok', 0)

    if not raw or str(raw).startswith('['):
        continue   # skip API-error / placeholder entries

    new_v, new_t, new_proof = parse_llm_response(raw, out_tok=out_tok, max_tok=_max_tok)

    # Normalise type string (same logic as pipeline cell)
    if new_v == 'True':
        t_lo = new_t.lower().strip()
        if 'copy' in t_lo:
            new_t_norm = 'Non-Shannon'
        elif 'double' in t_lo or 'markov' in t_lo:
            new_t_norm = 'Shannon'
        elif 'non-shannon' in t_lo or 'nonshannon' in t_lo:
            new_t_norm = 'Non-Shannon'
        elif 'shannon' in t_lo:
            new_t_norm = 'Shannon'
        else:
            new_t_norm = 'Unknown'
    else:
        new_t_norm = 'N/A'

    if old_v != new_v:
        _changed.append((r['num'], old_v, new_v, r.get('psitip', '?'), r['label'][:55]))

    r['llm']       = new_v
    r['ctype']        = new_t_norm
    r['llm_proof'] = new_proof or ''
    r['match']        = (r.get('psitip', '') == new_v)

# ── Report ────────────────────────────────────────────────────────────────────
if _changed:
    print(f"  {'#':>3}  {'Old':^9}  {'New':^9}  {'GT':^7}  Label")
    print(f"  {'─'*3}  {'─'*9}  {'─'*9}  {'─'*7}  {'─'*55}")
    for num, old, new, gt, lbl in _changed:
        ok = '✓' if gt == new else '✗'
        print(f"  {num:>3}  {old:^9}  {new:^9}  {gt:^7}  {ok} {lbl}")
else:
    print("  No verdicts changed.")

n_match = sum(1 for r in results_summary if r['match'])
tp  = sum(1 for r in results_summary if r.get('psitip')=='True'  and r['llm']=='True')
tn  = sum(1 for r in results_summary if r.get('psitip')=='False' and r['llm']=='False')
fp  = sum(1 for r in results_summary if r.get('psitip')=='False' and r['llm']=='True')
fn  = sum(1 for r in results_summary if r.get('psitip')=='True'  and r['llm']=='False')
unk = sum(1 for r in results_summary if r['llm'] == 'Unknown')
print(f"\nAfter re-parse: {n_match}/{len(results_summary)} match  "
      f"({100*n_match/len(results_summary):.1f}%)")

print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}  Unknown={unk}")


Re-parsing 75 records  (max_tok=10000) ...

  No verdicts changed.

After re-parse: 63/75 match  (84.0%)
  TP=42  TN=21  FP=4  FN=8  Unknown=0


In [15]:
# ── Cell 8: Re-score,  apply updated compare_proofs (no API calls) ───────────
# Run this cell after updating compare_proofs in cell 3.
# Requires results_summary from the previous run (cell 5).
#
# ⚠  IMPORTANT: Always run Cell 4 (improvements) AFTER Cell 3, BEFORE this cell.
#     Cell 4 applies broadened axiom patterns + LP boost. If you re-run Cell 3,
#     you MUST re-run Cell 4 too. The check below will catch this mistake.

if not getattr(compare_proofs, '_lp_boost', False):
    raise RuntimeError(
        "\n\nCell 4 (Scoring Improvements) must be run AFTER Cell 3 before rescoring!\n"
        "Correct order:  Cell 3  →  Cell 4  →  Cell 9 (this cell)\n"
        "Cell 3 was re-run and overrode the improved scoring functions.\n"
    )

print("Re-scoring stored proofs with current compare_proofs ...\n")

for r in results_summary:
    psitip_result = (r["psitip"] == "True")
    new_score, new_details, new_comps = compare_proofs(
        r["psitip_proof"], psitip_result, r["ptype"],
        r["llm_proof"], r["llm"],   r["ctype"],
        assumption=r.get("assumption", ""),
    )
    old_score = r["proof_score"]
    r["proof_score"] = new_score
    r["proof_components"] = new_comps

    old_str = f"{old_score:.0%}" if old_score is not None else "N/A"
    new_str = f"{new_score:.0%}" if new_score is not None else "N/A"
    delta   = ""
    if old_score is not None and new_score is not None:
        d = new_score - old_score
        delta = f"  ({'+' if d >= 0 else ''}{d:.0%})"
    print(f"  #{r['num']:>2}  {old_str:>5} → {new_str:>5}{delta}   {r['label'][:50]}")

# ── Updated summary table ─────────────────────────────────────
scored = [r for r in results_summary if r["proof_score"] is not None]
if scored:
    avg = sum(r["proof_score"] for r in scored) / len(scored)
    print(f"\n  Proof similarity (True cases): avg={avg:.0%}  over {len(scored)} case(s)")

print()
hdr = f"  {'#':>3}  {'PSITIP':>6}  {'P-Type':<14}  {'llm':>6}  {'C-Type':<14}  {'Match':>5}  {'Proof%':>6}"
print(hdr)
print(f"  {'─'*3}  {'─'*6}  {'─'*14}  {'─'*6}  {'─'*14}  {'─'*5}  {'─'*6}")
for r in results_summary:
    sym   = "✓" if r["match"] else "✗"
    ps    = f"{r['proof_score']:.0%}" if r["proof_score"] is not None else "  N/A"
    print(f"  {r['num']:>3}  {r['psitip']:>6}  {r['ptype']:<14}  {r['llm']:>6}"
          f"  {r['ctype']:<14}  {sym:>5}  {ps:>6}")


Re-scoring stored proofs with current compare_proofs ...

  # 1    88% →   88%  (+0%)   H(X)+H(Y)+H(Z) >= H(X,Y,Z)
  # 2    94% →   94%  (+0%)   H(X)+I(Y;Z|X) >= I(Y;Z)
  # 3    N/A →   N/A   H(X,Y) <= H(X)+H(Y)
  # 4    75% →   75%  (+0%)   X─Y─Z  =>  H(Y) >= I(X;Z)
  # 5    88% →   88%  (+0%)   H(Y|X)==0  =>  I(Y;Z) <= I(X;Z)
  # 6     0% →    0%  (+0%)   H(Y|X)==0  =>  H(Y,Z) <= H(X,Z)
  # 7   100% →  100%  (+0%)   H(Y|X)==0  =>  I(Y;Z|W) <= I(X;Z|W)
  # 8    67% →   67%  (+0%)   H(Y|X)==0 & H(Z|Y)==0  =>  I(Z;W) <= I(X;W)
  # 9   100% →  100%  (+0%)   H(Y|X)==0 & X─Y─Z  =>  H(Z|Y) >= H(Z|X)
  #10    N/A →   N/A   X─Y─Z  =>  I(X;Y) >= I(X;Z)
  #11   100% →  100%  (+0%)   X─Y─Z─W  =>  I(X;W) <= I(X;Y)
  #12   100% →  100%  (+0%)   X─Y─Z─W  =>  I(X;W) <= I(X;Z)
  #13    75% →   75%  (+0%)   X─Y─Z─W  =>  I(X;W) <= I(Y;W)
  #14   100% →  100%  (+0%)   X─Y─Z─W  =>  I(X;Z)+I(X;W) <= 2I(X;Y)
  #15   100% →  100%  (+0%)   X─Y─Z─W  =>  I(X;W) <= H(Y)
  #16    75% →   75%  (+0%)   X─Y─Z & H(W

In [16]:
# ── Cell 9: Save / load results_summary to/from JSON ───────────────────
# ═══════════════════════════════════════════════════════════════
#  SAVE / LOAD ,  persist and reload results_summary
# ═══════════════════════════════════════════════════════════════

def save_results(summary, path=None):
    """Serialise results_summary to JSON.  Auto-names by date+model if no path given."""
    if path is None:
        path = pathlib.Path(PROJECT_DIR) / f"results_{MODEL}_{datetime.date.today()}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(summary)} records → {path}")
    return path

def load_results(path=None):
    """Load results_summary from JSON.  If no path, loads the most-recent file."""
    if path is None:
        files = sorted(glob.glob(str(pathlib.Path(PROJECT_DIR) / f"results_{MODEL}_*.json")))
        if not files:
            raise FileNotFoundError(f"No results files found in {PROJECT_DIR}")
        path = files[-1]
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    print(f"Loaded {len(data)} records from {path}")
    return data

# ── Usage examples (uncomment as needed) ──────────────────────
# save_results(results_summary)
# results_summary = load_results()        # load most recent
# results_summary = load_results(r"C:\...\results_llm-sonnet-4-6_2025-01-01.json")


In [17]:

# ── Cell 10: Metrics,  verdict accuracy, type classification, proof quality ────
# ═══════════════════════════════════════════════════════════════════════════
#  ANALYSIS,  LLM-PSITIP Comparison Metrics
# ═══════════════════════════════════════════════════════════════════════════

assert results_summary, "Run cell 6 first."

# ── helpers ──────────────────────────────────────────────────────────────────
SEP  = "═" * 70
SEP2 = "─" * 70

def _pct(n, d):
    return f"{100*n/d:5.1f}%" if d else "   N/A"

def _prf4(tp, tn, fp, fn):
    n    = tp + tn + fp + fn
    acc  = (tp + tn) / n if n else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return acc, prec, rec, f1

def _print_metrics(tp, tn, fp, fn, indent=4):
    acc, prec, rec, f1 = _prf4(tp, tn, fp, fn)
    pad = " " * indent
    print(f"{pad}Accuracy  : {_pct(tp+tn, tp+tn+fp+fn)}   ({tp+tn}/{tp+tn+fp+fn})")
    print(f"{pad}Precision : {_pct(tp, tp+fp)}")
    print(f"{pad}Recall    : {_pct(tp, tp+fn)}")
    print(f"{pad}F1        : {f1:.4f}")
    print(f"{pad}Confusion : TP={tp}  TN={tn}  FP={fp}  FN={fn}")

def _wrong_table(rows, col1, col2):
    """Print a small mis-classification table."""
    if not rows:
        return
    print(f"    Misclassified ({len(rows)}):")
    print(f"    {'#':>3}  {'GT':^16}  {'Pred':^16}  Label")
    print(f"    {'─'*3}  {'─'*16}  {'─'*16}  {'─'*40}")
    for r in rows:
        print(f"    {r['num']:>3}  {col1(r):^16}  {col2(r):^16}  {r['label'][:40]}")

# ── dataset split ─────────────────────────────────────────────────────────────
true_r  = [r for r in results_summary if r["psitip"] == "True"]
false_r = [r for r in results_summary if r["psitip"] == "False"]

print(SEP)
print("  PIPELINE METRICS,  LLM-PSITIP Comparison")
print(SEP)
print(f"  Dataset: {len(results_summary)} cases  |  True = {len(true_r)}   False = {len(false_r)}")
type_cnt = Counter(r["ptype"] for r in true_r)
print("  Ground-truth type distribution (True cases):")
for tname, cnt in sorted(type_cnt.items()):
    print(f"    - {tname}: {cnt}")

# Verdict accuracy
print("\n" + SEP2)
print("  VERDICT ACCURACY")
print()
p1_tp  = sum(1 for r in results_summary if r["psitip"]=="True"  and r["llm"]=="True")
p1_tn  = sum(1 for r in results_summary if r["psitip"]=="False" and r["llm"]=="False")
p1_fp  = sum(1 for r in results_summary if r["psitip"]=="False" and r["llm"]=="True")
p1_fn  = sum(1 for r in results_summary if r["psitip"]=="True"  and r["llm"]=="False")
_print_metrics(p1_tp, p1_tn, p1_fp, p1_fn)
print()
print("  By type (True cases):")
for tname in ["Shannon", "Non-Shannon"]:
    sub = [r for r in true_r if r["ptype"] == tname]
    if sub:
        correct = sum(1 for r in sub if r["llm"] == "True")
        print(f"    - {tname}: {correct}/{len(sub)}  ({100*correct/len(sub):.1f}%)")
if false_r:
    correct = sum(1 for r in false_r if r["llm"] == "False")
    print(f"  By type (False cases):\n    - Specificity: {correct}/{len(false_r)} ({100*correct/len(false_r):.1f}%)")

wrong1 = sorted([r for r in results_summary if not r["match"]], key=lambda r: r["num"])
_wrong_table(wrong1,
             lambda r: f"{r['psitip']} ({r['ptype']})",
             lambda r: r["llm"])

# Type classification (True cases only)
_p2_sub = [r for r in results_summary if r["psitip"] == "True" and r["llm"] == "True"]
_p2_ran  = any(r["ctype"] not in ("Unknown", "N/A") for r in _p2_sub)
if _p2_sub and _p2_ran:
    print("\n" + SEP2)
    print("  TYPE CLASSIFICATION (True cases only)")
    print()
    print(f"    n = {len(_p2_sub)}  (Shannon = {sum(r['ptype']=='Shannon' for r in _p2_sub)}, Non-Shannon = {sum(r['ptype']=='Non-Shannon' for r in _p2_sub)})")
    p2_tp  = sum(1 for r in _p2_sub if r["ptype"] == "Shannon" and r["ctype"] == "Shannon")
    p2_tn  = sum(1 for r in _p2_sub if r["ptype"] == "Non-Shannon" and r["ctype"] != "Shannon")
    p2_fp  = sum(1 for r in _p2_sub if r["ptype"] == "Non-Shannon" and r["ctype"] == "Shannon")
    p2_fn  = sum(1 for r in _p2_sub if r["ptype"] == "Shannon" and r["ctype"] != "Shannon")
    _print_metrics(p2_tp, p2_tn, p2_fp, p2_fn)
    print()
    print(f"    - Shannon → predicted Shannon:      {p2_tp}/{{}} ({100*p2_tp/sum(r['ptype']=='Shannon' for r in _p2_sub):.1f}%)".format(sum(r['ptype']=='Shannon' for r in _p2_sub)))
    print(f"    - Non-Shannon → predicted Non-Shannon: {p2_tn}/{{}} ({100*p2_tn/sum(r['ptype']=='Non-Shannon' for r in _p2_sub):.1f}%)".format(sum(r['ptype']=='Non-Shannon' for r in _p2_sub)))

# Proof quality
print("\n" + SEP2)
print("  PROOF QUALITY (proof_score ∈ [0,1])")
print()
_scored = [r for r in results_summary if r["proof_score"] is not None]
if _scored:
    overall_avg = sum(r["proof_score"] for r in _scored) / len(_scored)
    print(f"    Scored cases: {len(_scored)} / {len(results_summary)}")
    print(f"    Overall avg : {overall_avg:.1%}")
    print()
    for tname in ["Shannon", "Non-Shannon"]:
        sub = [r for r in _scored if r["ptype"] == tname]
        if sub:
            avg = sum(r["proof_score"] for r in sub) / len(sub)
            print(f"    - {tname}: avg = {avg:.1%}   n = {len(sub)}")

# ── Per-component breakdown by type ──────────────────────────────────────
_comp_names = ['psitip_consec', 'axiom']
_comp_labels = {'psitip_consec': 'PV Consecutive', 'axiom': 'Axiom Recall'}
_has_comps = [r for r in _scored if r.get("proof_components")]
if _has_comps:
    print("\n" + SEP2)
    print("  PROOF COMPONENT BREAKDOWN BY TYPE")
    print()
    # Header
    _hdr = f"    {'Component':<14}"
    _types_present = []
    for tname in ["Shannon", "Non-Shannon"]:
        _tsub = [r for r in _has_comps if r["ptype"] == tname]
        if _tsub:
            _types_present.append((tname, _tsub))
            _hdr += f"  {tname:>14}"
    _hdr += f"  {'Overall':>14}"
    print(_hdr)
    print(f"    {'─'*14}" + f"  {'─'*14}" * (len(_types_present) + 1))

    # Per component row
    _weakest = {}  # {type: (comp_name, avg)}
    for comp in _comp_names:
        row = f"    {_comp_labels[comp]:<14}"
        for tname, _tsub in _types_present:
            vals = [r["proof_components"][comp] for r in _tsub
                    if comp in r["proof_components"] and r["proof_components"][comp] is not None]
            avg_c = sum(vals) / len(vals) if vals else None
            row += f"  {avg_c:>13.1%}" if avg_c is not None else f"  {'N/A':>13}"
            if avg_c is not None and (tname not in _weakest or avg_c < _weakest[tname][1]):
                _weakest[tname] = (comp, avg_c)
        all_vals = [r["proof_components"][comp] for r in _has_comps
                    if comp in r["proof_components"] and r["proof_components"][comp] is not None]
        avg_all = sum(all_vals) / len(all_vals) if all_vals else None
        row += f"  {avg_all:>13.1%}" if avg_all is not None else f"  {'N/A':>13}"
        print(row)

    # Total row
    row = f"    {'TOTAL':<14}"
    for tname, _tsub in _types_present:
        avg_t = sum(r["proof_score"] for r in _tsub) / len(_tsub)
        row += f"  {avg_t:>13.1%}"
    avg_overall = sum(r["proof_score"] for r in _has_comps) / len(_has_comps)
    row += f"  {avg_overall:>13.1%}"
    print(f"    {'─'*14}" + f"  {'─'*14}" * (len(_types_present) + 1))
    print(row)

    print()
    for tname, (wcomp, wavg) in _weakest.items():
        print(f"    ⚠ {tname} weakest: {_comp_labels[wcomp]} ({wavg:.1%})")

    # ── PV failure analysis: LP-contradicted vs LP-unproven ──────────────
    print("\n" + SEP2)
    print("  PV CONSECUTIVE FAILURE ANALYSIS")
    print()
    print("    LP-contradicted : Shannon LP proves the step is WRONG")
    print("                      (reversed direction also fails)")
    print("    LP-unproven     : LP cannot verify the step")
    print("                      (Shannon: likely a real error; Non-Shannon: may be valid,")
    print("                       LP is incomplete for non-Shannon reasoning)")
    print()

    _pv_fail_cases = [
        r for r in _has_comps
        if r["proof_components"].get("psitip_consec") is not None
        and r["proof_components"]["psitip_consec"] < 1.0
    ]

    # Aggregate per type
    _lp_agg = {}
    for tname, _tsub in _types_present:
        _tc = _tc_u = _tc_c = _cases_contra = _cases_unprov = _cases_mixed = 0
        _fail_cases = [r for r in _tsub
                       if r["proof_components"].get("psitip_consec") is not None
                       and r["proof_components"]["psitip_consec"] < 1.0]
        for r in _fail_cases:
            _assum = r.get("assumption") or r.get("constraints")
            _, _pv_v, _pv_t, _pvd = verify_proof_steps_psitip(
                r.get("llm_proof") or "", assumption=_assum)
            _c = sum(1 for d in _pvd if '[LP-contradicted]' in d)
            _u = sum(1 for d in _pvd if '[LP-unproven]'     in d)
            _tc_c += _c
            _tc_u += _u
            _tc   += _pv_t - _pv_v
            if   _c > 0 and _u == 0: _cases_contra  += 1
            elif _c == 0 and _u > 0: _cases_unprov  += 1
            elif _c > 0 and _u > 0:  _cases_mixed   += 1
        _lp_agg[tname] = dict(
            total_cases=len(_tsub),
            fail_cases=len(_fail_cases),
            fail_steps=_tc,
            contra_steps=_tc_c,
            unproven_steps=_tc_u,
            only_contra=_cases_contra,
            only_unproven=_cases_unprov,
            mixed=_cases_mixed,
        )

    # Print summary table
    _col = 16
    _hdr3  = f"    {'':28}  {'Shannon':>{_col}}  {'Non-Shannon':>{_col}}"
    _sep3  = f"    {'─'*28}  {'─'*_col}  {'─'*_col}"
    print(_hdr3)
    print(_sep3)

    def _lp_row(label, key):
        row3 = f"    {label:<28}"
        for tname, _ in _types_present:
            v = _lp_agg[tname][key]
            row3 += f"  {v:>{_col}}"
        print(row3)

    _lp_row("Cases with PV < 100%",       "fail_cases")
    _lp_row("  Total failed steps",        "fail_steps")
    _lp_row("  of which LP-contradicted",  "contra_steps")
    _lp_row("  of which LP-unproven",      "unproven_steps")
    print(_sep3)
    _lp_row("Case breakdown:",             "fail_cases")   # spacer label
    _lp_row("  only LP-contradicted cases","only_contra")
    _lp_row("  only LP-unproven cases",    "only_unproven")
    _lp_row("  mixed (both) cases",        "mixed")
    print()

    # Interpretation note per type
    for tname, _tsub in _types_present:
        ag = _lp_agg[tname]
        if ag["fail_cases"] == 0:
            print(f"    {tname}: no PV failures.")
            continue
        contra_pct = 100 * ag["contra_steps"] / ag["fail_steps"] if ag["fail_steps"] else 0
        unprov_pct = 100 * ag["unproven_steps"] / ag["fail_steps"] if ag["fail_steps"] else 0
        if tname == "Shannon":
            note = ("LP is COMPLETE → LP-contradicted steps are definite algebraic errors. "
                    "LP-unproven: LP cannot prove A<=B OR its reverse B<=A — llm combined "
                    "algebraic rearrangement + bounding into one step (should be split: "
                    "= to rearrange, then <=/>=  to bound).")
        else:
            note = ("LP is INCOMPLETE → LP-unproven steps may be correct non-Shannon "
                    "arguments (Copy Lemma etc.) that LP cannot verify. "
                    "LP-contradicted steps violate Shannon LP itself and warrant review.")
        print(f"    {tname}: {contra_pct:.0f}% of failed steps LP-contradicted, "
              f"{unprov_pct:.0f}% LP-unproven")
        print(f"      Note: {note}")
        print()

    # ── Per-case detail for lowest scoring cases ──────────────────────────
    print(SEP2)
    print("  LOWEST SCORING CASES (bottom 10)")
    print()
    _bottom = sorted(_has_comps, key=lambda r: r["proof_score"])[:10]
    _hdr2 = f"    {'#':>3}  {'Type':<14}  {'Score':>6}"
    for comp in _comp_names:
        _hdr2 += f"  {_comp_labels[comp]:>12}"
    _hdr2 += "  LP-contra  LP-unpvn  Label"
    print(_hdr2)
    print(f"    {'─'*3}  {'─'*14}  {'─'*6}" + f"  {'─'*12}" * len(_comp_names)
          + f"  {'─'*9}  {'─'*8}  {'─'*30}")
    for r in _bottom:
        _c2 = r.get("proof_components", {})
        line = f"    {r['num']:>3}  {r['ptype']:<14}  {r['proof_score']:>5.0%}"
        for comp in _comp_names:
            v = _c2.get(comp)
            if v is None:
                line += f"  {'N/A':>12}"
            else:
                marker = " ◄" if v < 0.50 else ""
                line += f"  {v:>10.0%}{marker}"
        # per-case LP tag counts
        _assum2 = r.get("assumption") or r.get("constraints")
        _pv2 = _c2.get("psitip_consec")
        if _pv2 is not None and _pv2 < 1.0:
            _, _, _, _pvd2 = verify_proof_steps_psitip(
                r.get("llm_proof") or "", assumption=_assum2)
            _nc = sum(1 for d in _pvd2 if '[LP-contradicted]' in d)
            _nu = sum(1 for d in _pvd2 if '[LP-unproven]' in d)
            line += f"  {_nc:>9}  {_nu:>8}"
        else:
            line += f"  {'-':>9}  {'-':>8}"
        line += f"  {r['label'][:30]}"
        print(line)

# Token usage
print("\n" + SEP2)
print("  TOKEN USAGE")
print()
_total_in  = sum(r["in_tok"]     for r in results_summary)
_total_out = sum(r["out_tok"]    for r in results_summary)
_cread     = sum(r.get("cache_read", 0) for r in results_summary)
print(f"    Input: {_total_in:,}   Output: {_total_out:,}   Cache-read: {_cread:,}")
print(SEP)


══════════════════════════════════════════════════════════════════════
  PIPELINE METRICS,  LLM-PSITIP Comparison
══════════════════════════════════════════════════════════════════════
  Dataset: 75 cases  |  True = 50   False = 25
  Ground-truth type distribution (True cases):
    - Non-Shannon: 25
    - Shannon: 25

──────────────────────────────────────────────────────────────────────
  VERDICT ACCURACY

    Accuracy  :  84.0%   (63/75)
    Precision :  91.3%
    Recall    :  84.0%
    F1        : 0.8750
    Confusion : TP=42  TN=21  FP=4  FN=8

  By type (True cases):
    - Shannon: 22/25  (88.0%)
    - Non-Shannon: 20/25  (80.0%)
  By type (False cases):
    - Specificity: 21/25 (84.0%)
    Misclassified (12):
      #         GT               Pred        Label
    ───  ────────────────  ────────────────  ────────────────────────────────────────
      6   True (Shannon)        False        H(Y|X)==0  =>  H(Y,Z) <= H(X,Z)
     22   True (Shannon)        False        X─Y─Z & Y─X─Z  =

                                           Shannon       Non-Shannon
    ────────────────────────────  ────────────────  ────────────────
    Cases with PV < 100%                         8                11
      Total failed steps                        15                23
      of which LP-contradicted                  12                21
      of which LP-unproven                       3                 2
    ────────────────────────────  ────────────────  ────────────────
    Case breakdown:                              8                11
      only LP-contradicted cases                 6                10
      only LP-unproven cases                     1                 0
      mixed (both) cases                         1                 1

    Shannon: 80% of failed steps LP-contradicted, 20% LP-unproven
      Note: LP is COMPLETE → LP-contradicted steps are definite algebraic errors. LP-unproven: LP cannot prove A<=B OR its reverse B<=A — llm combined algebraic rearrangement